# CrisisLens · Custom CNN Training (Kaggle GPU)

從頭訓練自建 CNN 做 5 類災情分類，並跑 3 個 ablation 對照（no-BN / big-FC / shallow）。

## 執行前
1. Notebook options → Accelerator → GPU T4 x2（或 P100）
2. Internet → On
3. Add data → `mikolajbabula/disaster-images-dataset-cnn-model`
4. 確認下方 Config cell 的 `DATA_DIR`

## 輸出
- `custom_cnn.pth` — v1 final 權重（~1.5 MB）
- `custom_cnn_classes.json` — 類別對照
- `training_curves.png`、`confusion_matrix.png`

In [ ]:
import sys, torch
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU 可用:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名稱:   {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️ 沒偵測到 GPU — 請從右側 Notebook options 啟用 GPU 後 Restart Session")

## 1. Config

In [ ]:
from pathlib import Path

DATA_DIR     = "/kaggle/input/disaster-images-dataset-cnn-model"
OUT_DIR      = Path("/kaggle/working")
WEIGHTS_PATH = OUT_DIR / "custom_cnn.pth"
MAPPING_PATH = OUT_DIR / "custom_cnn_classes.json"
CURVES_PATH  = OUT_DIR / "training_curves.png"
CM_PATH      = OUT_DIR / "confusion_matrix.png"

BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
EPOCHS        = 15
NUM_WORKERS   = 2
SEED          = 42
VAL_RATIO     = 0.2

FOLDER_TO_ZH = {
    "Damaged_Infrastructure": "地震或建築損壞",
    "Fire_Disaster":          "火災",
    "Land_Disaster":          "土石流或坍方",
    "Water_Disaster":         "淹水",
    "Non_Damage":             "其他或無明顯災害",
}

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

## 2. Imports

In [ ]:
import os, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. 確認資料集結構

In [ ]:
def find_data_root(base: str) -> str:
    base_path = Path(base)
    if not base_path.exists():
        raise FileNotFoundError(f"{base} 不存在 — 請確認資料集已 attach 並更新 DATA_DIR")
    expected = set(FOLDER_TO_ZH.keys())

    def matches(p: Path) -> bool:
        if not p.is_dir():
            return False
        subs = {x.name for x in p.iterdir() if x.is_dir()}
        return expected.issubset(subs)

    if matches(base_path):
        return str(base_path)
    for sub in base_path.rglob("*"):
        if matches(sub):
            return str(sub)
    raise FileNotFoundError(
        f"在 {base} 下找不到包含 {expected} 五個資料夾的層級。"
    )

DATA_ROOT = find_data_root(DATA_DIR)
print(f"✅ 資料根目錄: {DATA_ROOT}\n")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
print("類別數量：")
total_imgs = 0
for folder, zh in FOLDER_TO_ZH.items():
    fp = Path(DATA_ROOT) / folder
    n = sum(1 for p in fp.rglob("*") if p.suffix.lower() in IMG_EXTS) if fp.exists() else 0
    total_imgs += n
    mark = "✅" if n > 0 else "❌"
    print(f"  {mark} {folder:25s} → {zh:12s}  {n:5d} 張")
print(f"\n總圖片數: {total_imgs}")

## 4. Transforms (Train 用 augmentation，Val 用單純 resize)

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 5. Dataset & DataLoader

In [ ]:
full_ds     = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
classes_en  = full_ds.classes
classes_zh  = [FOLDER_TO_ZH.get(c, c) for c in classes_en]
num_classes = len(classes_en)

print("類別順序（ImageFolder 自動字母排序）：")
for i, (en, zh) in enumerate(zip(classes_en, classes_zh)):
    print(f"  [{i}] {en:25s} → {zh}")

n_total = len(full_ds)
n_val   = int(n_total * VAL_RATIO)
n_train = n_total - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
val_ds.dataset.transform = val_tf

def make_weighted_sampler(dataset, indices):
    targets = [dataset.targets[i] for i in indices]
    class_counts  = np.bincount(targets, minlength=num_classes)
    class_weights = 1.0 / np.maximum(class_counts.astype(float), 1.0)
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

sampler = make_weighted_sampler(full_ds, train_ds.indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"
Train: {n_train}  Val: {n_val}  Classes: {num_classes}")

## 6. v1 (Baseline) — 4-block CNN with BatchNorm + GAP

**⚠️ 注意**：這個 class 定義必須與本地 `models/custom_cnn_model.py` 的 `DisasterCNN_v1` 字字相同。
若架構不一致，訓練好的 `.pth` 無法在本地 streamlit 載入（layer name mismatch）。

In [ ]:
class DisasterCNN_v1(nn.Module):
    """4-block CNN baseline for 5-class disaster classification."""
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

### 訓練 v1（記錄訓練曲線，保存 best by Val Acc）

In [ ]:
def train_one_model(model_class, model_name: str, epochs: int = EPOCHS):
    """通用訓練函式 — 回傳 (history, best_val_acc, best_state, last_val_pred, last_val_true)。"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model     = model_class(num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn   = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler("cuda") if device == "cuda" else None

    history       = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc  = 0.0
    best_state    = None
    last_val_pred, last_val_true = [], []

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n=== {model_name} ===  可訓練參數: {n_params:,}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        # Train
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    logits = model(imgs)
                    loss   = loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = loss_fn(model(imgs), labels)
                loss.backward()
                optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # Validate
        model.eval()
        val_loss = 0.0
        correct = total_n = 0
        epoch_pred, epoch_true = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                logits = model(imgs)
                val_loss += loss_fn(logits, labels).item()
                preds = logits.argmax(1)
                correct += (preds == labels).sum().item()
                total_n += labels.size(0)
                epoch_pred.extend(preds.cpu().tolist())
                epoch_true.extend(labels.cpu().tolist())
        val_loss /= len(val_loader)
        val_acc   = correct / total_n

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        last_val_pred, last_val_true = epoch_pred, epoch_true

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())

        print(f"  Epoch {epoch:2d}/{epochs}  "
              f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  "
              f"Val Acc: {val_acc:.4f}  ({time.time()-t0:.1f}s)")

    print(f"  → Best Val Acc: {best_val_acc:.4f}")
    return history, best_val_acc, best_state, last_val_pred, last_val_true


# 訓練 v1
v1_history, v1_best_acc, v1_best_state, v1_pred, v1_true = train_one_model(
    DisasterCNN_v1, "v1 Baseline"
)

## 7. Ablation v2 — No-BN（拿掉所有 BatchNorm）

預期觀察：訓練 loss 震盪、收斂變慢、Val Acc 比 v1 下降 5-8%
學到什麼：BatchNorm 對訓練穩定性的關鍵作用

In [ ]:
class DisasterCNN_v2_NoBN(nn.Module):
    """v1 - all BatchNorm layers"""
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v2_history, v2_best_acc, v2_best_state, v2_pred, v2_true = train_one_model(
    DisasterCNN_v2_NoBN, "v2 No-BN"
)

## 8. Ablation v3 — Big-FC（GAP 換成 Flatten + 大 FC）

預期觀察：參數量 +100x、嚴重過擬合（Train Acc 99% / Val Acc ~50%）
學到什麼：GAP 為什麼取代 Flatten + 大 FC

In [ ]:
class DisasterCNN_v3_BigFC(nn.Module):
    """v1 with GAP replaced by Flatten + Linear(256*14*14, num_classes)"""
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256 * 14 * 14, num_classes),  # 50176 -> 5
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v3_history, v3_best_acc, v3_best_state, v3_pred, v3_true = train_one_model(
    DisasterCNN_v3_BigFC, "v3 Big-FC"
)

## 9. Ablation v4 — Shallow（只保留 Block 1 + Block 2）

預期觀察：Val Acc 比 v1 下降 10-15%
學到什麼：深度對特徵抽取能力的影響

In [ ]:
class DisasterCNN_v4_Shallow(nn.Module):
    """v1 with only Block 1 + Block 2 (drop Block 3 + Block 4)"""
    def __init__(self, num_classes: int = 5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v4_history, v4_best_acc, v4_best_state, v4_pred, v4_true = train_one_model(
    DisasterCNN_v4_Shallow, "v4 Shallow"
)

## 10. 四個變體的成績對比

In [ ]:
results = [
    ("v1 Baseline", v1_best_acc, sum(p.numel() for p in DisasterCNN_v1(num_classes).parameters())),
    ("v2 No-BN",    v2_best_acc, sum(p.numel() for p in DisasterCNN_v2_NoBN(num_classes).parameters())),
    ("v3 Big-FC",   v3_best_acc, sum(p.numel() for p in DisasterCNN_v3_BigFC(num_classes).parameters())),
    ("v4 Shallow",  v4_best_acc, sum(p.numel() for p in DisasterCNN_v4_Shallow(num_classes).parameters())),
]

print(f"{'Model':15s}  {'Val Acc':>8s}  {'Params':>12s}  {'vs v1':>8s}")
print("-" * 50)
for name, acc, n_params in results:
    delta = acc - v1_best_acc
    print(f"{name:15s}  {acc:>8.4f}  {n_params:>12,}  {delta:>+7.2%}")

## 11. 訓練曲線疊圖

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {"v1": "#38bdf8", "v2": "#f87171", "v3": "#fbbf24", "v4": "#a78bfa"}
for name, hist, key in [
    ("v1 Baseline", v1_history, "v1"),
    ("v2 No-BN",    v2_history, "v2"),
    ("v3 Big-FC",   v3_history, "v3"),
    ("v4 Shallow",  v4_history, "v4"),
]:
    ax1.plot(hist["val_loss"], color=colors[key], linewidth=2, label=name)
    ax2.plot(hist["val_acc"],  color=colors[key], linewidth=2, label=name, marker="o", markersize=3)

ax1.set_title("Validation Loss")
ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch"); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ 已存: {CURVES_PATH}")

## 12. v1 Confusion Matrix + Classification Report

In [ ]:
print("=== v1 Classification Report ===\n")
print(classification_report(
    v1_true, v1_pred,
    target_names=classes_zh,
    zero_division=0, digits=3,
))

cm = confusion_matrix(v1_true, v1_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes_zh, yticklabels=classes_zh, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("v1 Confusion Matrix")
plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CM_PATH, dpi=120, bbox_inches="tight")
plt.show()

## 13. 儲存 v1 (final deploy 版本) 權重

僅保存 v1 的 best state — v2/v3/v4 只在報告中比較，不 deploy。

In [ ]:
torch.save(v1_best_state, WEIGHTS_PATH)
print(f"✅ 權重已存: {WEIGHTS_PATH}  ({WEIGHTS_PATH.stat().st_size / 1e6:.2f} MB)")

mapping = {
    "classes":      classes_en,
    "zh_labels":    classes_zh,
    "class_to_idx": full_ds.class_to_idx,
    "num_classes":  num_classes,
    "architecture": "DisasterCNN_v1",
    "val_acc":      v1_best_acc,
}
with open(MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print(f"✅ 類別對照已存: {MAPPING_PATH}")
print(json.dumps(mapping, ensure_ascii=False, indent=2))

## 14. 下載連結

點下方連結下載；或從右側 Output 面板選 ⋮ → Download。

放到本地專案的 ：
-  → 
-  → 

訓練曲線、混淆矩陣是報告素材（可選下載）。

In [ ]:
from IPython.display import FileLink, display

print("可下載檔案：\n")
for path in [WEIGHTS_PATH, MAPPING_PATH, CURVES_PATH, CM_PATH]:
    if path.exists():
        size = path.stat().st_size / 1e6
        print(f"  {path.name:35s}  {size:6.2f} MB")
        display(FileLink(str(path)))